# HireMind — build the two human-review sheets (Colab, CPU is enough)

Makes the files for the two tasks that only people can do:

| File | For | What people fill in |
|---|---|---|
| `VAL_review_R1.xlsx`, `VAL_review_R2.xlsx` | two reviewers, **independently** | 150 VALIDATION pairs (50 per original class), **blind**: no original label and no model output shown. Label with the rubric + evidence span + reason + confidence. |
| `ERROR_tags.xlsx` | one person | 50 mistakes of the frozen model (M4 BERT-Tiny + Knowledge v1, seed 46), stratified by error type. Tick why it is wrong. |
| `VAL_answer_key_PRIVATE.csv` | coordinator only | case number ↔ row_uid ↔ original label. Do not give it to reviewers. |

All files contain CV and job text, so they are saved to a **private** Drive folder (`HireMind_PRIVATE_review_sheets`). Do not put them in the shared submission folder. Use the rubric `labeling_guidelines_v2.pdf` (A3).

The notebook also re-scores the frozen model on validation; it must give accuracy 56.97 (the frozen result), which checks that the predictions are the real ones.

In [ ]:
!pip -q install transformers==4.51.3 pandas==2.2.3 scikit-learn==1.6.1 huggingface-hub==0.30.2 openpyxl
%cd /content
!rm -rf _dl _s2 && git clone -q https://github.com/Hams770/HireMind-DL.git _dl && git -C _dl checkout -q 1f22f34462c400f884bb0a00f061dbea0bd687d5
!git clone -q https://github.com/Hams770/Senior-2.git _s2 && git -C _s2 checkout -q ec28dfc85e618a737cb192cbf959714cfee89aa5
!wget -q -O m4_s46.pt https://github.com/HIREMINDPROJECT/HireMind-Plan/raw/main/weights/a1_knowledge_M4_seed46_weights.pt
!echo "0953ce892ee0f075d37c1e6b4ff0bceb02a7932fddbca0ee5954f5e17fcb0c8e  m4_s46.pt" | sha256sum -c

In [ ]:
import os, sys, numpy as np, pandas as pd, torch
os.environ["HIREMIND_MAX_LEN"] = "512"
sys.path.insert(0, "/content/_dl/src"); sys.path.insert(0, "/content/_s2/evaluation/knowledge")
from transformers import AutoTokenizer
from hiremind_dl import load_config
from hiremind_dl.data import as_dataset, download_raw, load_manifest, read_raw, split_frame, tokenize
from hiremind_dl.models import build_model
from hiremind_dl.evaluation import evaluate
from knowledge import features, knowledge_line
cfg = load_config()
raw = {"train": read_raw(download_raw(cfg, "train"))}   # the test file is never downloaded
man = load_manifest(cfg)
va = split_frame(raw, man, "validation", cfg["dataset"]["rows"]["validation"])
assert np.bincount(va.label_id).tolist() == [579, 294, 289]
vk = va.copy()
vk["job_description_text"] = [knowledge_line(features(cv, job)) + "\n" + job for cv, job in zip(va.resume_text, va.job_description_text)]
tok = AutoTokenizer.from_pretrained(cfg["student"]["name"], revision=cfg["student"]["revision"])
ds = as_dataset(tokenize(vk, tok, 512))
m = build_model("M0", cfg, 46); m.load_state_dict(torch.load("m4_s46.pt", map_location="cpu")); m.eval()
_, logits, pred = evaluate(m, ds, "cpu", 64)
prob = logits.softmax(-1).numpy(); pred = np.asarray(pred)
acc = 100 * (pred == va.label_id.to_numpy()).mean()
print(f"frozen model validation accuracy: {acc:.2f} (expected 56.97)")
assert abs(acc - 56.97) < 0.05, "predictions do not match the frozen model"

In [ ]:
CL = ["No Fit", "Potential Fit", "Good Fit"]
rng = np.random.default_rng(20261124)
# ---- A) blind VALIDATION review: 50 per original class
pick = np.concatenate([rng.choice(np.where(va.label_id.to_numpy() == c)[0], 50, replace=False) for c in range(3)])
rng.shuffle(pick)
key = pd.DataFrame({"case_no": np.arange(1, 151), "row_uid": va.row_uid.to_numpy()[pick], "original_label_D0": [CL[i] for i in va.label_id.to_numpy()[pick]]})
sheet = pd.DataFrame({"case_no": key.case_no, "job_description": va.job_description_text.to_numpy()[pick], "cv_text": va.resume_text.to_numpy()[pick],
                      "your_label (Good Fit / Potential Fit / No Fit)": "", "evidence_span (copy the CV words)": "", "reason": "", "confidence (1-5)": ""})
# ---- B) error tags: 50 mistakes of the frozen model, spread over the 6 error types
y = va.label_id.to_numpy(); wrong = np.where(pred != y)[0]
types = sorted({(y[i], pred[i]) for i in wrong})
per = {t: [i for i in wrong if (y[i], pred[i]) == t] for t in types}
take = []
while len(take) < 50 and any(per.values()):
    for t in types:
        if per[t] and len(take) < 50:
            j = rng.integers(len(per[t])); take.append(per[t].pop(j))
take = np.array(take)
err = pd.DataFrame({"case_no": np.arange(1, len(take) + 1), "row_uid": va.row_uid.to_numpy()[take], "true_label": [CL[i] for i in y[take]],
                    "model_prediction": [CL[i] for i in pred[take]], "model_confidence": prob[take].max(1).round(2),
                    "job_description": va.job_description_text.to_numpy()[take], "cv_text": va.resume_text.to_numpy()[take],
                    "label_looks_wrong (y/n)": "", "extraction_or_missing_evidence (y/n)": "", "semantic_mismatch (y/n)": "",
                    "evidence_cut_by_length (y/n)": "", "other (write)": "", "note": ""})
print("error types sampled:", pd.Series(list(zip(err.true_label, err.model_prediction))).value_counts().to_dict())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/HireMind_PRIVATE_review_sheets"; os.makedirs(D, exist_ok=True)
for r in ("R1", "R2"):
    with pd.ExcelWriter(f"{D}/VAL_review_{r}.xlsx") as w:
        pd.DataFrame({"instructions": [
            "Label each pair with the rubric (labeling_guidelines_v2.pdf): Good Fit / Potential Fit / No Fit.",
            "Work alone; do not look at the other reviewer's file.",
            "Copy the CV words that support your label into evidence_span; give a one-line reason and a confidence 1-5.",
            "Not mentioned in the CV is not proof the candidate lacks it."]}).to_excel(w, sheet_name="READ_ME", index=False)
        sheet.to_excel(w, sheet_name="cases", index=False)
with pd.ExcelWriter(f"{D}/ERROR_tags.xlsx") as w:
    pd.DataFrame({"instructions": [
        "Each row is a pair the frozen model got wrong. Read the job and the CV.",
        "Tick y for every reason that applies: the label looks wrong; the evidence was missed or wrongly extracted; the meaning was misread; the evidence was cut by the 512-token limit; other.",
        "Use the note for anything else."]}).to_excel(w, sheet_name="READ_ME", index=False)
    err.to_excel(w, sheet_name="cases", index=False)
key.to_csv(f"{D}/VAL_answer_key_PRIVATE.csv", index=False)
print("saved to", D)